In [1]:
from pydantic import BaseModel, Field
from typing import List

class PermissionFinding(BaseModel):
    title: str = Field(description="A short, clear title for the finding.")
    severity: str = Field(description="Must be 'info', 'review', or 'high concern'.")
    explanation: str = Field(description="Detailed explanation of the privacy risk.")
    evidence: List[str] = Field(description="The specific raw permissions involved.")
    limitations: List[str] = Field(description="What cannot be proven from the manifest alone.")
    recommendation: str = Field(description="Actionable steps for the user.")

In [6]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load the .env file automatically
load_dotenv()

# Initialize the model (Groq is perfect here for speed)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# FORCE the model to only output your Pydantic structure
agent_brain = llm.with_structured_output(PermissionFinding)

In [7]:
from langchain_core.prompts import ChatPromptTemplate

purpose_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a mobile privacy Purpose Analyst. 
    Your job is to compare an app's stated purpose with its extracted permissions.
    If a permission is disproportionate to the app's purpose, flag it.
    You must always state the limitations of your analysis to prevent unsupported claims."""),
    ("user", "App Purpose: {app_purpose}\nExtracted Permissions: {permissions}")
])

# Connect the prompt to the structured LLM
purpose_analyst_agent = purpose_prompt | agent_brain

Mock Permissions


In [8]:
# 1. Fake the data Role A will eventually send you
mock_app_purpose = "A simple flashlight app that turns on the camera flash."
mock_permissions = [
    "android.permission.CAMERA",
    "android.permission.ACCESS_FINE_LOCATION",
    "android.permission.READ_CONTACTS"
]

# 2. Run your agent using the fake data
result = purpose_analyst_agent.invoke({
    "app_purpose": mock_app_purpose, 
    "permissions": mock_permissions
})

# 3. See if the AI accurately outputs your Pydantic format!
print(result)

title='Unnecessary Location Permission' severity='high concern' explanation='The app’s stated purpose is to control the camera flash for a flashlight function. Access to fine location is not required for turning on the flash, nor is it typically used by flashlight apps. Granting this permission could allow the app to track the user’s location, which is a privacy risk disproportionate to the app’s purpose.' evidence=['android.permission.ACCESS_FINE_LOCATION'] limitations=['The manifest alone does not reveal whether the app uses location for background services, analytics, or other features not disclosed in the purpose statement. The app could also be a bundled app that includes additional functionality not reflected in the stated purpose.'] recommendation='Review the app’s privacy policy and source code if available. If the location permission is not used, consider removing it or requesting the user to deny it. If the permission is required for legitimate background functionality, the a

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from agents.shared_llm import agent_brain

permission_mapper_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a mobile security Permission Mapper.
Your task is to analyze technical Android permission strings and translate them into clear, human-readable definitions.

Rules:
1. Explain in plain English what specific hardware (e.g., camera lens), personal data (e.g., GPS coordinates, contact book), or background capabilities the permission grants.
2. Evaluate the inherent sensitivity of the requested access.
3. Keep explanations clear, objective, and technical without assuming malicious intent.
4. State explicit limitations (e.g., declaring a permission in the manifest allows access, but doesn't prove active data collection at runtime)."""),
    ("user", "Extracted Permissions to Map: {permissions}")
])

permission_mapper_agent = permission_mapper_prompt | agent_brain

In [14]:
%load_ext autoreload
%autoreload 2
from agents.permission_mapper import permission_mapper_agent

# Mock permissions list
mock_permissions = [
    "android.permission.CAMERA",
    "android.permission.ACCESS_FINE_LOCATION",
    "android.permission.READ_CONTACTS"
]

# Run the Permission Mapper
mapper_result = permission_mapper_agent.invoke({
    "permissions": mock_permissions
})

print("Title:", mapper_result.title)
print("Severity:", mapper_result.severity)
print("Explanation:", mapper_result.explanation)
print("Evidence:", mapper_result.evidence)
print("Limitations:", mapper_result.limitations)
print("Recommendation:", mapper_result.recommendation)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Title: Camera Access Permission
Severity: review
Explanation: Allows the app to access the device’s camera hardware, enabling it to capture photos or record video. The permission does not specify which camera (front or rear) or how the media will be used, but it grants the ability to take images or video at any time the app is running.
Evidence: ['android.permission.CAMERA']
Limitations: ['Declaring the permission in the manifest does not prove that the app actually uses the camera; it only indicates that the app has the capability to do so. Runtime checks or user prompts are required to confirm active usage.']
Recommendation: Review the app’s privacy policy and usage logs to confirm whether camera access is necessary for its functionality. If not needed, consider removing the permission or requesting it only when required.


In [1]:
from agents.combination_analyst import combination_analyst_agent

mock_app_purpose = "A simple flashlight app that turns on the camera flash."
mock_permissions = [
    "android.permission.CAMERA",
    "android.permission.ACCESS_FINE_LOCATION",
    "android.permission.READ_CONTACTS"
]

combination_result = combination_analyst_agent.invoke({
    "app_purpose": mock_app_purpose,
    "permissions": mock_permissions
})

print(combination_result)

title='Camera, Location, and Contacts Permissions Combined' severity='high concern' explanation='When an app declares CAMERA, ACCESS_FINE_LOCATION, and READ_CONTACTS together, it gains the ability to capture images, tag them with precise GPS coordinates, and cross‑reference those images with the user’s contact list. Individually, CAMERA only lets the app take photos, ACCESS_FINE_LOCATION only provides location data, and READ_CONTACTS only reads the contact list. Combined, the app can build a detailed profile: it can take a photo of a person, determine where that photo was taken, and then match the person to a contact entry (including name, phone number, email, and photo). This correlation enables a range of privacy‑violating scenarios:\n\n1. **Location‑based contact profiling** – The app can map each contact to the places the user visits, revealing patterns such as frequent restaurants, workplaces, or friends’ homes.\n2. **Facial recognition of contacts** – By capturing images of conta

In [2]:
from agents.skeptical_reviewer import skeptical_reviewer_agent

# Pass the result from your previous combination analyst test
reviewed_result = skeptical_reviewer_agent.invoke({
    "app_purpose": mock_app_purpose,
    "findings": str(combination_result)
})

print("Title:", reviewed_result.title)
print("Severity:", reviewed_result.severity)
print("Explanation:", reviewed_result.explanation)
print("Limitations:", reviewed_result.limitations)

Title: Camera, Location, and Contacts Permissions Combined
Severity: review
Explanation: The manifest shows that the app declares CAMERA, ACCESS_FINE_LOCATION, and READ_CONTACTS. While a flashlight app only needs CAMERA (or possibly VIBRATE) to control the camera flash, the presence of the other two permissions does not automatically imply malicious intent. Many apps bundle third‑party libraries—such as advertising SDKs, analytics services, or optional features—that require location or contact access. For example, an ad network might request location to serve geo‑targeted ads, and a social‑sharing SDK might request contacts to find friends. If the flashlight app includes such optional modules or future‑proofs for additional features, the permissions could be justified. Without runtime evidence of how the app uses these permissions, the manifest alone only indicates potential capability.

The risk of correlating photos, location, and contacts is real, but it is not a certainty that the 

available groq models

In [5]:
import os, requests
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

response = requests.get(
    "https://api.groq.com/openai/v1/models", 
    headers={"Authorization": f"Bearer {api_key}"}
)

# Print all models currently accessible on your account
models = [m["id"] for m in response.json().get("data", [])]
print("Available Groq Models:", models)

Available Groq Models: ['allam-2-7b', 'meta-llama/llama-prompt-guard-2-86m', 'canopylabs/orpheus-arabic-saudi', 'qwen/qwen3.8-27b', 'meta-llama/llama-prompt-guard-2-22m', 'whisper-large-v3-turbo', 'openai/gpt-oss-safeguard-20b', 'whisper-large-v3', 'openai/gpt-oss-120b', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-20b']
